In [10]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

Pulling Data from Database

In [11]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'
engine = create_engine(url, echo=False)

In [12]:
with engine.begin() as conn: 
    result = conn.execute(text(f'SET search_path TO {pg_schema};'))

In [14]:
with engine.begin() as conn: # Done with echo=False
    data_airports = conn.execute(text(f'''
                               SELECT * FROM mart_faa_stats; 
                                '''))
    data_flights = conn.execute(text(f'''
                               SELECT * FROM mart_rout_stats; 
                                '''))
    data_weather = conn.execute(text(f'''
                               SELECT * FROM mart_selected_faa_stats_weather; 
                                '''))    

### Let's create a dataframe out of that
df_airports = pd.DataFrame(data_airports.all()) 
df_flights = pd.DataFrame(data_flights.all())
df_weather = pd.DataFrame(data_weather.all())

Working with Data 

In [21]:
display(df_airports.head())
display(df_airports.info())

,faa,city,country,numer_of_unique_dep_connections,numer_of_unique_arr_connections,total_planed_dep,total_planed_arr,total_canceled_dep,total_canceled_arr,total_diverted_dep,total_diverted_arr,total_actual_dep,total_actual_arr
0,ATL,Atlanta,United States,147,5,53236,4320,982,82,122,6,52254,4238
1,DEN,Denver,United States,157,5,45694,3923,2786,165,141,4,42908,3758
2,DTW,Detroit,United States,90,5,19751,2904,516,75,32,2,19235,2829
3,JFK,New York,United States,65,5,21892,3436,378,36,66,3,21514,3400
4,LAX,Los Angeles,United States,90,5,31001,5196,695,96,74,9,30306,5100


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   faa                              6 non-null      object
 1   city                             6 non-null      object
 2   country                          6 non-null      object
 3   numer_of_unique_dep_connections  6 non-null      int64 
 4   numer_of_unique_arr_connections  6 non-null      int64 
 5   total_planed_dep                 6 non-null      int64 
 6   total_planed_arr                 6 non-null      int64 
 7   total_canceled_dep               6 non-null      int64 
 8   total_canceled_arr               6 non-null      int64 
 9   total_diverted_dep               6 non-null      int64 
 10  total_diverted_arr               6 non-null      int64 
 11  total_actual_dep                 6 non-null      int64 
 12  total_actual_arr                 6 non-n

None

In [19]:
display(df_flights.head())
display(df_flights.info())

,origin,origin_city,origin_country,dest,dest_city,dest_country,total_planed_flights_on_rout,unique_airlines_on_rout,unique_planes_on_rout,avg_actual_elapsed_time_on_rout,avg_arr_delay,max_arr_delay,min_arr_delay,total_canceled_on_rout,total_diverted_on_rout
0,ATL,Atlanta,United States,DEN,Denver,United States,946,4,493,0 days 03:24:57.576419,0 days 00:17:03.537118,0 days 10:25:00,-1 days +23:20:00,30,0
1,ATL,Atlanta,United States,DTW,Detroit,United States,842,3,416,0 days 01:50:48.317308,0 days 00:09:21.274038,0 days 10:31:00,-1 days +23:27:00,10,0
2,ATL,Atlanta,United States,JFK,New York,United States,655,2,343,0 days 02:12:17.647059,0 days 00:20:10.959752,0 days 16:32:00,-1 days +23:29:00,9,0
3,ATL,Atlanta,United States,LAX,Los Angeles,United States,833,3,376,0 days 04:59:14.087591,0 days 00:13:37.518248,0 days 06:22:00,-1 days +23:13:00,10,1
4,ATL,Atlanta,United States,ORD,Chicago,United States,1054,8,555,0 days 02:03:43.123181,0 days 00:14:12.861300,1 days 00:25:00,-1 days +23:26:00,23,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype          
---  ------                           --------------  -----          
 0   origin                           30 non-null     object         
 1   origin_city                      30 non-null     object         
 2   origin_country                   30 non-null     object         
 3   dest                             30 non-null     object         
 4   dest_city                        30 non-null     object         
 5   dest_country                     30 non-null     object         
 6   total_planed_flights_on_rout     30 non-null     int64          
 7   unique_airlines_on_rout          30 non-null     int64          
 8   unique_planes_on_rout            30 non-null     int64          
 9   avg_actual_elapsed_time_on_rout  30 non-null     timedelta64[ns]
 10  avg_arr_delay                    30 non-null     tim

None

In [20]:
display(df_weather.head())
display(df_weather.info())

,faa,flight_date,numer_of_unique_dep_connections,total_planed_dep,total_canceled_dep,total_diverted_dep,total_actual_dep,numer_of_unique_arr_connections,total_planed_arr,total_canceled_arr,...,cw,day_part,temp_c,dewpoint_c,humidity_perc,precipitation_mm,wind_direction,wind_speed_kmh,pressure_hpa,condition_code
0,ATL,2022-12-01,139,911,2,0,909,5,73,0,...,48.0,night,10.0,-2.6,41.0,0.0,110,9.4,1030.6,3
1,ATL,2022-12-01,139,911,2,0,909,5,73,0,...,48.0,evening,12.8,-4.3,30.0,0.0,150,11.2,1030.0,1
2,ATL,2022-12-01,139,911,2,0,909,5,73,0,...,48.0,evening,13.3,-4.8,28.0,0.0,150,13.0,1029.7,1
3,ATL,2022-12-01,139,911,2,0,909,5,73,0,...,48.0,evening,13.3,-6.8,24.0,0.0,160,9.4,1029.9,2
4,ATL,2022-12-01,139,911,2,0,909,5,73,0,...,48.0,evening,12.2,-6.7,26.0,0.0,160,5.4,1030.1,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8928 entries, 0 to 8927
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   faa                              8928 non-null   object 
 1   flight_date                      8928 non-null   object 
 2   numer_of_unique_dep_connections  8928 non-null   int64  
 3   total_planed_dep                 8928 non-null   int64  
 4   total_canceled_dep               8928 non-null   int64  
 5   total_diverted_dep               8928 non-null   int64  
 6   total_actual_dep                 8928 non-null   int64  
 7   numer_of_unique_arr_connections  8928 non-null   int64  
 8   total_planed_arr                 8928 non-null   int64  
 9   total_canceled_arr               8928 non-null   int64  
 10  total_diverted_arr               8928 non-null   int64  
 11  total_actual_arr                 8928 non-null   int64  
 12  date                

None